# Stage 2: Parse Reasoning Traces into Steps

Takes the full reasoning traces from Stage 1 and splits them into individual steps.

Each trace looks like:
```
Step 1: Natalia sold 48 clips in April.
Step 2: In May she sold half as many: 48/2 = 24.
Step 3: Total = 48 + 24 = 72.
#### 72
```
We split this into one row per step, removing the `Step N:` prefix and the `####` line. These individual steps become the nodes in Stage 4's graphs.

**Input:** `data/gsm8k_with_traces.csv`  
**Output:** `data/gsm8k_steps.csv`

## Cell 1 — Install libraries

Stage 2 only needs `pandas` and `re` — both are already installed from Stage 1.  
This cell is just a safety check to make sure everything is available.

After this cell → **Kernel → Restart** → run all cells top to bottom.

In [1]:
import sys
!{sys.executable} -m pip install -q pandas numpy --user
print(" Libraries ready")

 Libraries ready


## Cell 2 — Imports and file paths

We import the libraries and define two file paths:
- `INPUT_FILE` — the CSV from Stage 1 (one row per trace)
- `OUTPUT_FILE` — the new CSV we create here (one row per step)

Both files live in the same `data/` folder as Stage 1.

In [2]:
import os
import re
import pandas as pd
import numpy as np

# File paths — same data/ folder as Stage 1
DATA_DIR    = os.path.join(os.getcwd(), 'data')
INPUT_FILE  = os.path.join(DATA_DIR, 'gsm8k_with_traces.csv')   # Stage 1 output
OUTPUT_FILE = os.path.join(DATA_DIR, 'gsm8k_steps.csv')          # Stage 2 output

print(f" Imports OK")
print(f" Data folder:  {DATA_DIR}")
print(f" Input file:   {INPUT_FILE}")
print(f" Output file:  {OUTPUT_FILE}")

# Check input file exists
if os.path.exists(INPUT_FILE):
    print(f"\n Input file found")
else:
    print(f"\n Input file NOT found — make sure Stage 1 is complete first")

 Imports OK
 Data folder:  C:\Users\Aruna\Desktop\latthika_research\cot_project\data
 Input file:   C:\Users\Aruna\Desktop\latthika_research\cot_project\data\gsm8k_with_traces.csv
 Output file:  C:\Users\Aruna\Desktop\latthika_research\cot_project\data\gsm8k_steps.csv

 Input file found


## Cell 3 — Load and inspect the Stage 1 data

We load `gsm8k_with_traces.csv` and print a summary so we can see what we're working with.

We also print one real trace so you can see exactly what the raw text looks like before we parse it.

In [3]:
df = pd.read_csv(INPUT_FILE)

print(f"Rows loaded:    {len(df)}")
print(f"Columns:        {list(df.columns)}")
print()

# Label distribution
counts = df['label'].value_counts()
print(f"Label distribution:")
print(f"  Correct (1): {counts.get(1, 0)} traces")
print(f"  Wrong   (0): {counts.get(0, 0)} traces")
print()

# Split distribution
print(f"Split distribution:")
for s in ['train', 'val', 'test']:
    print(f"  {s}: {len(df[df['split']==s])}")
print()

# Print one real trace so we can see what we're parsing
sample = df[df['label'] == 1].iloc[0]
print("=" * 60)
print("EXAMPLE TRACE (label=1, correct):")
print("=" * 60)
print(f"Question: {sample['question'][:100]}...")
print()
print(f"Raw trace text:")
print(sample['trace'])
print("=" * 60)

Rows loaded:    1948
Columns:        ['orig_idx', 'split', 'question', 'gt_answer', 'trace', 'pred', 'gt', 'label', 'num_steps']

Label distribution:
  Correct (1): 1461 traces
  Wrong   (0): 487 traces

Split distribution:
  train: 1363
  val: 292
  test: 293

EXAMPLE TRACE (label=1, correct):
Question: When Nathan is cold, he adds an extra blanket to his bed. Each blanket warms him up by 3 degrees. On...

Raw trace text:
Step 1: First, let's find out how many blankets Nathan added to his bed. Since he added half of the 14 blankets, we divide 14 by 2.
14 / 2 = 7

Step 2: Each blanket warms up Nathan by 3 degrees. To find the total temperature increase, we multiply the number of blankets added by the temperature increase per blanket.
7 * 3 = 21

#### 21


## Cell 4 — Define the parsing and cleaning functions

This is the core of Stage 2. Two functions:

**`parse_trace(trace_text)`**  
Takes one big trace string and splits it into a list of steps.  
It looks for the pattern `Step N:` and cuts the text there.  
The `#### answer` line is automatically dropped since it doesn't start with `Step`.

**`clean_step(step_text)`**  
Takes one step string and cleans it:  
- Removes the `Step N:` prefix (the number adds no meaning for the embedder)  
- Strips extra whitespace from both ends  
- Returns `None` if the step is empty after cleaning (so we can skip it)

In [4]:
def parse_trace(trace_text):
    """
    Split a trace string into a list of individual step strings.
    
    Example input:
        'Step 1: She has 48 clips.\nStep 2: Half of 48 = 24.\nStep 3: Total = 72.\n#### 72'
    
    Example output:
        ['Step 1: She has 48 clips.', 'Step 2: Half of 48 = 24.', 'Step 3: Total = 72.']
    """
    if not trace_text or pd.isna(trace_text):
        return []
    
    # Split on 'Step N:' pattern — works for Step 1:, Step 2:, Step 10:, etc.
    # re.split keeps the delimiter when we use a capturing group ()
    parts = re.split(r'(Step\s+\d+:\s*)', str(trace_text))
    
    steps = []
    i = 0
    while i < len(parts):
        # re.split gives us alternating [text, delimiter, text, delimiter, ...]
        # We want to combine each delimiter with the text that follows it
        if re.match(r'Step\s+\d+:\s*', parts[i]):
            # This part is the 'Step N:' header
            step_header = parts[i]
            step_body   = parts[i+1] if i+1 < len(parts) else ''
            full_step   = step_header + step_body
            # Drop the #### answer line if it ended up inside a step body
            full_step = re.sub(r'####.*', '', full_step).strip()
            if full_step:
                steps.append(full_step)
            i += 2
        else:
            # Text before the first Step — usually empty, skip it
            i += 1
    
    # Fallback: if no Step N: pattern found, treat the whole trace as one step
    # (after removing the #### line)
    if len(steps) == 0:
        cleaned = re.sub(r'####.*', '', str(trace_text)).strip()
        if cleaned:
            steps = [cleaned]
    
    return steps


def clean_step(step_text):
    """
    Clean one step string:
    - Remove the 'Step N:' prefix
    - Strip whitespace
    - Return None if empty after cleaning
    
    Example input:  'Step 2: Half of 48 is 24 clips.'
    Example output: 'Half of 48 is 24 clips.'
    """
    if not step_text:
        return None
    
    # Remove 'Step N:' prefix
    cleaned = re.sub(r'^Step\s+\d+:\s*', '', step_text.strip())
    
    # Strip any leftover whitespace
    cleaned = cleaned.strip()
    
    # Return None if nothing left
    return cleaned if cleaned else None


#  Quick test on a real example 
test_trace = """Step 1: Natalia sold 48 clips in April.
Step 2: In May she sold half as many, so 48 / 2 = 24 clips.
Step 3: Total = 48 + 24 = 72 clips.
#### 72"""

print("TEST — parsing a sample trace:")
print("-" * 50)
parsed = parse_trace(test_trace)
print(f"Number of steps found: {len(parsed)}")
print()
for j, step in enumerate(parsed):
    cleaned = clean_step(step)
    print(f"Step {j+1} raw:     {step}")
    print(f"Step {j+1} cleaned: {cleaned}")
    print()

print(" Functions working correctly")

TEST — parsing a sample trace:
--------------------------------------------------
Number of steps found: 3

Step 1 raw:     Step 1: Natalia sold 48 clips in April.
Step 1 cleaned: Natalia sold 48 clips in April.

Step 2 raw:     Step 2: In May she sold half as many, so 48 / 2 = 24 clips.
Step 2 cleaned: In May she sold half as many, so 48 / 2 = 24 clips.

Step 3 raw:     Step 3: Total = 48 + 24 = 72 clips.
Step 3 cleaned: Total = 48 + 24 = 72 clips.

 Functions working correctly


## Cell 5 — Process all traces into steps

Now we apply the two functions to every trace in the dataset.

For each trace we:
1. Parse it into a list of steps
2. Clean each step
3. Create one output row per step, keeping track of:
   - Which trace it came from (`trace_id`)
   - Which position it was in (`step_num`)
   - The trace's label — correct or wrong (`label`)
   - Which split it belongs to — train/val/test (`split`)

This loops over all rows — runs in a few seconds, no API calls needed.

In [5]:
all_steps = []   # Will hold one dict per step
skipped   = 0    # Traces we couldn't parse (missing trace text)

for trace_id, row in df.iterrows():
    
    # Skip rows with no trace text
    if pd.isna(row['trace']) or str(row['trace']).strip() == '':
        skipped += 1
        continue
    
    # Step 1: Parse trace into list of steps
    steps = parse_trace(row['trace'])
    
    # Step 2: Clean each step and save as a row
    valid_steps = []
    for step in steps:
        cleaned = clean_step(step)
        if cleaned:                      # Skip empty steps
            valid_steps.append(cleaned)
    
    # Skip traces that produced zero valid steps
    if len(valid_steps) == 0:
        skipped += 1
        continue
    
    # Step 3: Save each step as one row in all_steps
    for step_num, step_text in enumerate(valid_steps, start=1):
        all_steps.append({
            'trace_id':    trace_id,              # Links back to the parent trace
            'orig_idx':    row['orig_idx'],        # Original GSM8K index
            'split':       row['split'],           # train / val / test
            'question':    row['question'],        # Original question
            'step_num':    step_num,               # Position: 1, 2, 3...
            'step_text':   step_text,              # Cleaned step text
            'total_steps': len(valid_steps),       # Total steps in this trace
            'label':       int(row['label']),      # 1 = correct, 0 = wrong
        })

# Convert to DataFrame
df_steps = pd.DataFrame(all_steps)

print(f"Input traces:    {len(df)}")
print(f"Skipped traces:  {skipped} (missing or unparseable)")
print(f"Output steps:    {len(df_steps)}")
print(f"Avg steps/trace: {len(df_steps) / max(len(df) - skipped, 1):.1f}")
print()
print(f"Label distribution in steps:")
step_counts = df_steps['label'].value_counts()
print(f"  Steps from correct traces (1): {step_counts.get(1, 0)}")
print(f"  Steps from wrong traces   (0): {step_counts.get(0, 0)}")
print()
print(" All traces parsed")

Input traces:    1948
Skipped traces:  0 (missing or unparseable)
Output steps:    9490
Avg steps/trace: 4.9

Label distribution in steps:
  Steps from correct traces (1): 6400
  Steps from wrong traces   (0): 3090

 All traces parsed


## Cell 6 — Save the steps CSV

Save the output to `data/gsm8k_steps.csv`.  
This file is the input for Stage 3 (embedding each step into a 384-number vector).

In [6]:
df_steps.to_csv(OUTPUT_FILE, index=False)

print(f" Saved to: {OUTPUT_FILE}")
print(f"   Rows: {len(df_steps)}")
print(f"   Columns: {list(df_steps.columns)}")
print()

# Preview first 5 rows
print("Preview (first 5 rows):")
print("-" * 80)
preview = df_steps[['trace_id', 'split', 'step_num', 'step_text', 'total_steps', 'label']].head(5)
for _, r in preview.iterrows():
    print(f"trace_id={r['trace_id']}  step={r['step_num']}/{r['total_steps']}  label={r['label']}  split={r['split']}")
    print(f"  text: {r['step_text'][:80]}")
    print()

 Saved to: C:\Users\Aruna\Desktop\latthika_research\cot_project\data\gsm8k_steps.csv
   Rows: 9490
   Columns: ['trace_id', 'orig_idx', 'split', 'question', 'step_num', 'step_text', 'total_steps', 'label']

Preview (first 5 rows):
--------------------------------------------------------------------------------
trace_id=0  step=1/2  label=1  split=train
  text: First, let's find out how many blankets Nathan added to his bed. Since he added 

trace_id=0  step=2/2  label=1  split=train
  text: Each blanket warms up Nathan by 3 degrees. To find the total temperature increas

trace_id=1  step=1/3  label=1  split=train
  text: Determine the total amount of ground beef Maurice has purchased.
Maurice bought 

trace_id=1  step=2/3  label=1  split=train
  text: Calculate how many 2-pound burgers Maurice can make with the ground beef.
Each b

trace_id=1  step=3/3  label=1  split=train
  text: Since Maurice also wants a burger, subtract 1 from the total number of burgers t



## Cell 7 — Quality checks

Six checks to confirm the output is correct before moving to Stage 3.

| # | Check | Target |
|---|---|---|
| 1 | Output file exists | Yes |
| 2 | No empty step texts | 0 empty |
| 3 | Step numbers are sequential | Step 1, 2, 3... per trace |
| 4 | All 3 splits present | train, val, test |
| 5 | Average steps per trace | 2 to 10 |
| 6 | Both labels present | At least some 0s and 1s |

In [7]:
df_check = pd.read_csv(OUTPUT_FILE)

print("=" * 55)
print("  STAGE 2 QUALITY CHECKS")
print("=" * 55)

all_pass = True

# Check 1: File exists and has rows
print(f"\n[1] Total step rows: {len(df_check)}")
if len(df_check) > 0:
    print("     PASS")
else:
    print("     FAIL — no rows")
    all_pass = False

# Check 2: No empty step texts
empty = df_check['step_text'].isna().sum() + (df_check['step_text'].str.strip() == '').sum()
print(f"\n[2] Empty step texts: {empty}")
if empty == 0:
    print("     PASS")
else:
    print(f"      {empty} empty steps found")
    all_pass = False

# Check 3: Step numbers are sequential per trace
bad_sequences = 0
for tid, group in df_check.groupby('trace_id'):
    expected = list(range(1, len(group) + 1))
    actual   = sorted(group['step_num'].tolist())
    if expected != actual:
        bad_sequences += 1
print(f"\n[3] Traces with non-sequential step numbers: {bad_sequences}")
if bad_sequences == 0:
    print("     PASS")
else:
    print(f"      {bad_sequences} traces have step numbering issues")
    all_pass = False

# Check 4: All 3 splits present
splits_found = set(df_check['split'].unique())
splits_need  = {'train', 'val', 'test'}
print(f"\n[4] Splits present: {splits_found}")
split_counts = df_check['split'].value_counts()
for s in ['train', 'val', 'test']:
    print(f"    {s}: {split_counts.get(s, 0)} steps")
if splits_need.issubset(splits_found):
    print("     PASS")
else:
    print(f"     FAIL — missing splits: {splits_need - splits_found}")
    all_pass = False

# Check 5: Average steps per trace is sensible
avg_steps = df_check.groupby('trace_id')['step_num'].max().mean()
print(f"\n[5] Average steps per trace: {avg_steps:.1f}")
min_steps = df_check.groupby('trace_id')['step_num'].max().min()
max_steps = df_check.groupby('trace_id')['step_num'].max().max()
print(f"    Min: {min_steps}  Max: {max_steps}")
if 2 <= avg_steps <= 10:
    print("     PASS")
else:
    print("      Unusual step count — check parsing")
    all_pass = False

# Check 6: Both labels present
labels_found = set(df_check['label'].unique())
n_correct = len(df_check[df_check['label']==1])
n_wrong   = len(df_check[df_check['label']==0])
print(f"\n[6] Labels in steps:")
print(f"    Steps from correct traces (1): {n_correct}")
print(f"    Steps from wrong traces   (0): {n_wrong}")
if 0 in labels_found and 1 in labels_found:
    print("     PASS")
else:
    print("     FAIL — only one label type found")
    all_pass = False

print()
print("=" * 55)
if all_pass:
    print("   ALL CHECKS PASSED — Stage 2 complete!")
    print("    Ready for Stage 3 (embedding steps)")
else:
    print("    SOME CHECKS FAILED — see above")
print("=" * 55)

  STAGE 2 QUALITY CHECKS

[1] Total step rows: 9490
     PASS

[2] Empty step texts: 0
     PASS

[3] Traces with non-sequential step numbers: 0
     PASS

[4] Splits present: {'val', 'test', 'train'}
    train: 6614 steps
    val: 1407 steps
    test: 1469 steps
     PASS

[5] Average steps per trace: 4.9
    Min: 2  Max: 21
     PASS

[6] Labels in steps:
    Steps from correct traces (1): 6400
    Steps from wrong traces   (0): 3090
     PASS

   ALL CHECKS PASSED — Stage 2 complete!
    Ready for Stage 3 (embedding steps)


## Cell 8 — Manual inspection

We look at real examples — one correct trace and one wrong trace — to confirm the parsing looks right.

For the wrong trace, you should be able to spot the error in one of the steps by reading it.

In [8]:
df_check = pd.read_csv(OUTPUT_FILE)

def print_trace_steps(trace_id, df):
    """Print all steps for one trace."""
    steps = df[df['trace_id'] == trace_id].sort_values('step_num')
    first = steps.iloc[0]
    label_text = "CORRECT " if first['label'] == 1 else "WRONG "
    print(f"trace_id={trace_id}  label={first['label']} ({label_text})  split={first['split']}")
    print(f"Q: {first['question'][:90]}...")
    print(f"Steps ({first['total_steps']} total):")
    for _, step in steps.iterrows():
        print(f"  Step {step['step_num']}: {step['step_text'][:100]}")
    print()

# Pick one correct trace and one wrong trace
correct_traces = df_check[df_check['label'] == 1]['trace_id'].unique()
wrong_traces   = df_check[df_check['label'] == 0]['trace_id'].unique()

print("=" * 65)
print("EXAMPLE 1 — Correct trace (label=1)")
print("=" * 65)
if len(correct_traces) > 0:
    print_trace_steps(correct_traces[0], df_check)

print("=" * 65)
print("EXAMPLE 2 — Wrong trace (label=0)")
print("=" * 65)
if len(wrong_traces) > 0:
    print_trace_steps(wrong_traces[0], df_check)
else:
    print("No wrong traces found — check label column")

EXAMPLE 1 — Correct trace (label=1)
trace_id=0  label=1 (CORRECT )  split=train
Q: When Nathan is cold, he adds an extra blanket to his bed. Each blanket warms him up by 3 d...
Steps (2 total):
  Step 1: First, let's find out how many blankets Nathan added to his bed. Since he added half of the 14 blank
  Step 2: Each blanket warms up Nathan by 3 degrees. To find the total temperature increase, we multiply the n

EXAMPLE 2 — Wrong trace (label=0)
trace_id=2  label=0 (WRONG )  split=train
Q: Rica's group won in a dance competition. She got 3/8 of the prize money. From Rica's prize...
Steps (7 total):
  Step 1: Let's define the total prize money as 'x'. Rica got 3/8 of the prize money, which means she received
  Step 2: Rica spent 1/5 of her prize money, which is (1/5)*(3/8)*x. To find out how much she spent, we need t
  Step 3: Rica is left with $300 after spending 1/5 of her prize money. We can set up an equation to represent
  Step 4: We can simplify the equation by first finding a co

##  Stage 2 Complete!

**Output file:** `data/gsm8k_steps.csv`

| Column | What it contains |
|---|---|
| `trace_id` | Which trace this step belongs to |
| `orig_idx` | Original GSM8K row index |
| `split` | train / val / test |
| `question` | The original maths problem |
| `step_num` | Position of this step (1, 2, 3...) |
| `step_text` | Cleaned text of just this step |
| `total_steps` | Total steps in the parent trace |
| `label` | 1 = correct trace, 0 = wrong trace |

---

**What just happened:**  
Each trace (1 row) → multiple steps (N rows, one per step)  
The `####` answer line was dropped  
The `Step N:` prefix was removed from each step text  
All splits and labels were carried forward correctly  

**Next → Stage 3:**  
Each `step_text` gets converted into a **384-number vector** using the `all-MiniLM-L6-v2` sentence embedding model.  
These vectors become the **node features** in our reasoning graphs.